## Feed Forward Neural Network

Welcome, everyone! Today, we are stepping into the mathematical machinery that powers modern artificial intelligence: **Feed Forward Neural Networks (FFNNs)**.

I know that making the leap from introductory calculus and linear algebra to neural networks can seem daunting. It often looks like a black box of magic. However, as mathematicians, you already possess the tools to understand this: matrices, vectors, and the multivariate chain rule.

Let's demystify the black box by exploring forward propagation, deriving the backpropagation algorithm from scratch, and solving a classic non-linear puzzle: the XOR gate.

### 1. The Architecture and Forward Propagation

A Feed Forward Neural Network passes data in one direction—from inputs, through intermediate "hidden" layers, to an output.

For our case study, we will build a network with:

* An input vector $x \in \mathbb{R}^2$.
* One hidden layer with 4 neurons.
* One output layer with 1 neuron.

Each connection between neurons has a **weight**, and each neuron has a **bias**. Every neuron sums its inputs and passes the result through an **activation function** to introduce non-linearity. We will use the **sigmoid function**:


$$\sigma(z) = \frac{1}{1 + e^{-z}}$$

Its derivative, which we will need later, is elegantly simple:


$$\sigma'(z) = \sigma(z)(1 - \sigma(z))$$

#### The Forward Pass Equations

Let $W^{(1)}$ be the weight matrix of shape $(2, 4)$ and $b^{(1)}$ be the bias vector for the hidden layer. Let $W^{(2)}$ and $b^{(2)}$ be the weights and bias for the output layer.

The forward propagation algorithm is a sequence of linear transformations followed by element-wise non-linear activations:

1. **Hidden layer pre-activation:**

$$z^{(1)} = x W^{(1)} + b^{(1)}$$


2. **Hidden layer activation:**

$$a^{(1)} = \sigma(z^{(1)})$$


3. **Output layer pre-activation:**

$$z^{(2)} = a^{(1)} W^{(2)} + b^{(2)}$$


4. **Final Output prediction:**

$$\hat{y} = a^{(2)} = \sigma(z^{(2)})$$

### 2. The Loss Function and Gradient Descent

To teach the network, we must quantify how wrong its predictions are using a **Loss Function**. For this lecture, we will use the **Least-Squares Loss** (or Mean Squared Error) for a single training example with true label $y$:

$$L = \frac{1}{2}(\hat{y} - y)^2$$

Our objective is to find the matrices $W^{(1)}, W^{(2)}$ and vectors $b^{(1)}, b^{(2)}$ that minimize $L$. We do this using **Gradient Descent**. We calculate the gradient of the loss with respect to each parameter and take a small step in the opposite direction.

If $\alpha$ is our learning rate, the update rules are:


$$W^{(i)} \leftarrow W^{(i)} - \alpha \frac{\partial L}{\partial W^{(i)}}$$

$$b^{(i)} \leftarrow b^{(i)} - \alpha \frac{\partial L}{\partial b^{(i)}}$$

### 3. Backward Propagation: Deriving the Gradients



How do we find these partial derivatives? We use the **chain rule** of calculus, working backward from the output to the input. This is **backpropagation**.

### Step 3a: Gradients for the Output Layer

Let's find how the loss changes with respect to the output weights $W^{(2)}$. By the chain rule:


$$\frac{\partial L}{\partial W^{(2)}} = \frac{\partial L}{\partial \hat{y}} \cdot \frac{\partial \hat{y}}{\partial z^{(2)}} \cdot \frac{\partial z^{(2)}}{\partial W^{(2)}}$$

Let's compute the individual parts:

1. **Derivative of Loss wrt output:** $\displaystyle\frac{\partial L}{\partial \hat{y}} = (\hat{y} - y)$


2. **Derivative of activation wrt pre-activation:** $\displaystyle\frac{\partial \hat{y}}{\partial z^{(2)}} = \sigma'(z^{(2)}) = \hat{y}(1 - \hat{y})$

Let's define the "error signal" at the output node as $\delta^{(2)}$:


$$\delta^{(2)} = \frac{\partial L}{\partial z^{(2)}} = (\hat{y} - y) \odot \sigma'(z^{(2)})$$


*(Note: $\odot$ denotes element-wise multiplication).*

Now, because $z^{(2)} = a^{(1)} W^{(2)} + b^{(2)}$, the explicit partial derivatives for the output layer parameters are:


$$\frac{\partial L}{\partial W^{(2)}} = (a^{(1)})^T \delta^{(2)}$$

$$\frac{\partial L}{\partial b^{(2)}} = \delta^{(2)}$$

### Step 3b: Gradients for the Hidden Layer

To update $W^{(1)}$ and $b^{(1)}$, we must push the error signal back through the network.


$$\delta^{(1)} = \frac{\partial L}{\partial z^{(1)}} = \left( \delta^{(2)} (W^{(2)})^T \right) \odot \sigma'(z^{(1)})$$

Using this hidden error signal, the explicit partial derivatives for the hidden layer parameters are:


$$\frac{\partial L}{\partial W^{(1)}} = x^T \delta^{(1)}$$

$$\frac{\partial L}{\partial b^{(1)}} = \delta^{(1)}$$

*(When training with a batch of $N$ examples simultaneously, we average these gradients over the $N$ examples).*

### 4. Example: The XOR Gate Problem

The XOR (Exclusive OR) gate takes two binary inputs and outputs 1 if they are strictly different, and 0 if they are the same.

| Input 1 | Input 2 | Output |
| --- | --- | --- |
| 0 | 0 | 0 |
| 0 | 1 | 1 |
| 1 | 0 | 1 |
| 1 | 1 | 0 |

If you plot these points on a 2D Cartesian plane, you will see that you cannot draw a single straight line separating the 1s from the 0s. A simple linear regression model completely fails here. However, by projecting the data into a 4-dimensional space using our hidden layer, the network bends the space to make the data linearly separable.

Here is a pure `numpy` realization using functions. Pay close attention to how the code directly mirrors our mathematical derivations from Section 3.

In [1]:
import numpy as np

def sigmoid(z):
    """Sigmoid activation function."""
    return 1.0 / (1.0 + np.exp(-z))

def sigmoid_derivative(a):
    """Derivative of the sigmoid function, given the activated output 'a'."""
    return a * (1.0 - a)

def train_xor_network(X, y, hidden_neurons=4, epochs=10000, learning_rate=0.5):
    """Trains a feed forward neural network on the XOR problem."""
    # Seed for reproducibility
    np.random.seed(42)
    
    # Number of training examples
    N = X.shape[0]
    
    # 1. Initialize weights and biases randomly
    # W1: 2 inputs -> 4 hidden neurons
    W1 = np.random.uniform(-1, 1, (2, hidden_neurons))
    b1 = np.zeros((1, hidden_neurons))
    
    # W2: 4 hidden neurons -> 1 output
    W2 = np.random.uniform(-1, 1, (hidden_neurons, 1))
    b2 = np.zeros((1, 1))
    
    print("Starting training phase...\n")
    
    for epoch in range(epochs):
        # --- FORWARD PROPAGATION ---
        z1 = np.dot(X, W1) + b1
        a1 = sigmoid(z1)
        
        z2 = np.dot(a1, W2) + b2
        y_hat = sigmoid(z2)
        
        # --- LOSS COMPUTATION (Least-Squares) ---
        loss = np.mean(0.5 * (y_hat - y)**2)
        
        if epoch % 2000 == 0 or epoch == epochs - 1:
            print(f"Epoch {epoch:5d} | Loss: {loss:.6f}")
            
        # --- BACKWARD PROPAGATION ---
        # Error signal at output (delta^2)
        delta2 = (y_hat - y) * sigmoid_derivative(y_hat)
        
        # Gradients for output layer (averaged over batch N)
        dW2 = np.dot(a1.T, delta2) / N
        db2 = np.sum(delta2, axis=0, keepdims=True) / N
        
        # Error signal at hidden layer (delta^1)
        delta1 = np.dot(delta2, W2.T) * sigmoid_derivative(a1)
        
        # Gradients for hidden layer
        dW1 = np.dot(X.T, delta1) / N
        db1 = np.sum(delta1, axis=0, keepdims=True) / N
        
        # --- GRADIENT DESCENT UPDATE ---
        W1 -= learning_rate * dW1
        b1 -= learning_rate * db1
        W2 -= learning_rate * dW2
        b2 -= learning_rate * db2
        
    return W1, b1, W2, b2, y_hat

# --- Execute Case Study ---
if __name__ == "__main__":
    # XOR dataset
    X = np.array([[0, 0], 
                  [0, 1], 
                  [1, 0], 
                  [1, 1]])
                  
    y = np.array([[0], 
                  [1], 
                  [1], 
                  [0]])

    # Train the network
    W1, b1, W2, b2, predictions = train_xor_network(X, y, epochs=10000, learning_rate=1.0)
    
    print("\nTraining Complete. Final Predictions:")
    for i in range(len(X)):
        print(f"Input: {X[i]} | Target: {y[i][0]} | Predicted: {predictions[i][0]:.4f}")

Starting training phase...

Epoch     0 | Loss: 0.131458
Epoch  2000 | Loss: 0.002924
Epoch  4000 | Loss: 0.000893
Epoch  6000 | Loss: 0.000509
Epoch  8000 | Loss: 0.000352
Epoch  9999 | Loss: 0.000268

Training Complete. Final Predictions:
Input: [0 0] | Target: 0 | Predicted: 0.0145
Input: [0 1] | Target: 1 | Predicted: 0.9760
Input: [1 0] | Target: 1 | Predicted: 0.9773
Input: [1 1] | Target: 0 | Predicted: 0.0290


Notice how the loss decreases over time, eventually dropping near zero, at which point the predictions strongly align with the target XOR outputs.